# Coherence of the forest chips - from the public seasonal coherence data set

Part one of the coherence test (written down 2026-10-04 before it was built). This notebook makes, for every training and validation chip of the forest dataset, the radar coherence of the same place as an image of 128 by 128 pixels and saves it to Drive. Notebook 04b then scores it. No GPU is needed.

## What it does

- the forest dataset's training and validation splits on the runtime disk, as notebook 03 takes them
- where a chip sits. ForTy gives one latitude and longitude per chip and does not say whether it is the centre or the upper left corner. For 300 training chips (seed 0) the chip's own VV backscatter, smoothed to 90 m, is set against the data set's summer VV backscatter read under both readings, the correlation per chip, and the reading with the higher median is used. A reading shifted by 1.3 km is printed next to them as the floor. No label is used
- the coherence of every chip, the 12-day VV coherence of the four seasons of December 2019 to November 2020, per pixel the mean over the seasons with data, 0 to 1, interpolated from the 3 arcsec grid at every chip pixel. Chips without coherence get 0 and are counted
- the arrays and a record of how they were made go to Drive

## Data

The Global Seasonal Sentinel-1 Interferometric Coherence and Backscatter Data Set (Kellndorfer et al., Scientific Data 9, 73, 2022), CC BY 4.0, public on AWS without an account. Only the windows a chip covers are read, about 15 rows of a tile, never whole tiles. The forest dataset comes from its private Hugging Face copy when DATASET_HF_OWNER is set, else from its copy on Drive under MyDrive/datasets. HF_TOKEN is never printed.

## Runtime and time

The CPU runtime with High-RAM, for its 8 cores (the reading waits on the network, 32 chips at once). A few minutes, an estimate, the cell prints its chips per second.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/coherence/forty_v1/. train_coherence.npy and val_coherence.npy (uint8, 0 to 255 for coherence 0 to 1, in the chip order of the prepared splits) and coherence.json with the reading used, the check's numbers and the counts. A split whose file is on Drive is not made again.

## How to run

Run the cells from top to bottom. The last cell flushes Drive and then releases the Colab machine (set END_RUNTIME = False in that cell to keep the runtime), and a cell that fails stops the run before it.


In [ ]:
# Mount Google Drive. The data and the results are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. rasterio when the runtime lacks it, tokens, folders, the forest dataset on the runtime disk.
import subprocess
import sys

for module in ("rasterio",):                       # installed only when the runtime lacks it
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", module], check=True)
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sartransfer import env
from sartransfer.distil import stage0
from sartransfer.domains import hub, prepared
from sartransfer.domains.prepared import load_manifest
from sartransfer.domains import coherence

start = time.perf_counter()
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the private data copies")
DATA_TOKEN = os.environ.get("HF_WRITE_TOKEN") or HF_TOKEN   # the private dataset copies, as notebook 03
DATASET_OWNER = os.environ.get("DATASET_HF_OWNER", "").strip()   # optional, without it the Drive copy is used

# The test, fixed before it was built
DATASET = "forty_v1"
SPLITS = ("train", "val")
NEEDED = SPLITS
CHECK_CHIPS, CHECK_SEED = 300, 0                   # the training chips of the placement check

RESULTS_FOLDER = "sar-transfer-distillation"     # the Drive folder this notebook writes to, a new name re-runs it from scratch
CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
assert MYDRIVE.is_dir(), "Google Drive is not mounted, run the mount cell first"   # no model weights here, no cache
PREP = CONTENT / "sar-transfer-distillation" / "prepared"          # runtime disk, as notebook 03
OUT = MYDRIVE / RESULTS_FOLDER / "coherence" / DATASET   # the results of this notebook
RECORD = OUT / "coherence.json"
TIMES = {}                       # seconds of every step of this run, by name


def took(since, name):
    """The time of a cell as text, recorded under `name`."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def on_disk(name):
    """True when the manifest and the train and val files of a dataset are on the runtime disk."""
    if not (PREP / name / "manifest.json").is_file():
        return False
    return all((PREP / name / file).is_file() for file in prepared.split_files(NEEDED))


if on_disk(DATASET):
    print(f"{DATASET}  on the runtime disk already")
else:
    clock, source = time.perf_counter(), None
    if DATASET_OWNER:
        try:                                       # first the private Hugging Face copy, train and val
            hub.fetch(DATASET, DATASET_OWNER, PREP, token=DATA_TOKEN, splits=NEEDED)
            source = "Hugging Face"
        except Exception as err:
            print(f"{DATASET}  not on Hugging Face ({type(err).__name__}, {hide(err)[:200]})")
    if source is None:                             # then the copy on Drive
        copy = MYDRIVE / "datasets" / hub.repo_name(DATASET)
        assert (copy / "manifest.json").is_file(), f"{DATASET} is neither on Hugging Face nor on Drive under {copy}"
        prepared.copy_prepared(copy, PREP, DATASET, splits=NEEDED)
        source = "Drive"
    print(f"{DATASET}  from {source} in {(time.perf_counter() - clock) / 60:.1f} min")
man = load_manifest(PREP, DATASET)
print(f"{DATASET:<10} splits {man['splits']}, {man['chip_size']} px, units {man['units']}")
OUT.mkdir(parents=True, exist_ok=True)
print(f"CPU cores {os.cpu_count()}, results go to {OUT}")
print(took(start, "setup"))


In [ ]:
# Placement. Which reading of the chip's latitude and longitude fits (coherence.placement_check) - the chip's own
# VV in dB against the data set's summer VV backscatter, per chip, on 300 training chips. The higher median of
# centre and corner is used, the shifted reading is the floor. A record on Drive from an earlier run is reused.
start = time.perf_counter()
TILES = coherence.Tiles()
done = json.loads(RECORD.read_text(encoding="utf-8")) if RECORD.is_file() else {}
if "placement" in done:
    placement = done["placement"]
    print("the placement of an earlier run, from coherence.json")
else:
    meta = prepared.load_split(PREP, DATASET, "train")
    vv_db = np.asarray(meta["images"][:, 0], np.float32)    # VV in dB, the first channel of the prepared split
    placement = coherence.placement_check(TILES, vv_db, meta["meta"]["lat"].to_numpy(),
                                          meta["meta"]["lon"].to_numpy(), n=CHECK_CHIPS, seed=CHECK_SEED)
    done["placement"] = placement
    RECORD.write_text(json.dumps(done, indent=1), encoding="utf-8")
for reading, value in placement["medians"].items():
    print(f"  {reading:<24} median correlation {value:+.3f}")
ANCHOR = placement["anchor"]
print(f"the chip latitude and longitude are read as the {ANCHOR}, from {placement['chips']} training chips")
print(took(start, "placement"))


In [ ]:
# Coherence. Every chip of the train and val splits, 32 chips at once (coherence.WORKERS, the reason is written
# there), the chips per second printed as it goes. A split whose file is on Drive is not made again.
start = time.perf_counter()
for split in SPLITS:
    target = OUT / f"{split}_coherence.npy"
    if target.is_file() and split in done.get("splits", {}):
        print(f"{split:<5} on Drive already, {done['splits'][split]['chips']:,} chips")
        continue
    meta = prepared.load_split(PREP, DATASET, split)["meta"]
    print(f"{split:<5} {len(meta):,} chips")
    arrays, record = coherence.build(TILES, meta["lat"].to_numpy(), meta["lon"].to_numpy(), ANCHOR)
    np.save(target, arrays)
    done.setdefault("splits", {})[split] = record
    print(f"{split:<5} {record['chips']:,} chips in {record['seconds'] / 60:.1f} min, {record['chips_per_second']:.1f} "
          f"chips per second, {record['tile_windows_read']:,} tile windows, {record['chips_without_coherence']} chips "
          f"without coherence, {100 * record['pixels_without_coherence_share']:.2f} percent of pixels without")
done.update(source=coherence.SOURCE, layer=coherence.COHERENCE, seasons=list(coherence.SEASONS), anchor=ANCHOR,
            encoding="uint8, 0 to 255 for coherence 0 to 1, no data as 0", times=TIMES)
RECORD.write_text(json.dumps(done, indent=1), encoding="utf-8")
print(f"saved {RECORD}")
print(took(start, "coherence"))


In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
